# Whitening Makes the Noise as Loud as the Signal

**Topic:** PCA whitening &nbsp;·&nbsp; **Difficulty:** Medium

PCA whitening divides each principal component by its standard deviation so every
direction has unit variance. On data that is genuinely low rank, that is exactly
the problem: 32 of the 40 directions are pure numerical noise, each with variance about
`1e-12`, so between them they carry a `2.4e-14` share of the total variance, and whitening promotes every one of them to an
equal partner of the real signal.

Nothing overflows. The damage is subtler: after whitening, 80% of the total
variance lives in directions that contain no information at all, and whatever you
fit downstream will happily use them.

## The data (created in the next cell)

- `X` `(600, 40)`: genuinely 8-dimensional data plus tiny numerical noise in
  the other 32 directions.
- Eigenvalues span more than 14 orders of magnitude.

## Your task

With **NumPy only**:

1. Centre `X`, form the sample covariance with denominator `n - 1` (`np.cov(Xc, rowvar=False)`), and take `eigh`. Sort descending. The checks are tight enough to reject a denominator of `n`.
2. `naive_white`: project onto all 40 components and divide by `sqrt(lambda)`.
3. `reg_white`: the same but dividing by `sqrt(lambda + 1e-5)`.
4. `trunc_white`: keep only components with `lambda > 1e-8` and whiten those.
5. For the raw projection and for `naive_white` and `reg_white`, report the
   fraction of total variance carried by components 8 onwards, the directions
   that are pure noise.
6. Set `answers` to a dict with `"eigvals"` (40, descending), `"n_kept"` (int),
   `"noise_frac_raw"`, `"noise_frac_naive"`, `"noise_frac_reg"`, and
   `"cov_err_trunc"`: `||cov(trunc_white) - I||_F` on the kept dimensions.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks naive whitening promotes the noise directions to most of the variance,
that regularized whitening does not, and that the truncated version really is
white.

In [ ]:
import numpy as np

rng = np.random.default_rng(3007)
n, d, r = 600, 40, 8
B = rng.normal(0, 1, (r, d))
Z = rng.normal(0, 1, (n, r)) * np.linspace(3.0, 1.0, r)
X = Z @ B + rng.normal(0, 1e-6, (n, d))
print(f"X {X.shape}, true rank {r} plus 1e-6 numerical noise")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("eigvals", [40], 1323.6870723620548, 76.50075105326323, 1e-06),
    ("n_kept", [], 8.0, -1.3395041185159933, 1e-06),
    ("noise_frac_raw", [], 2.3832838967298025e-14, -3.99052324407802e-15, 1e-06),
    # The whitened noise columns inherit the rounding error of the tail
    # eigenvalues, so this fraction is stable only to about 1 percent: 1e-2
    # here and 1e-6 on the other keys.
    ("noise_frac_naive", [], 0.8037956302835519, -0.13458594465124707, 1e-02),
    ("noise_frac_reg", [], 3.9434011760647524e-07, -6.602752645374434e-08, 1e-06),
    ("cov_err_trunc", [], 3.5246551516431957e-15, -5.901612614968342e-16, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "noise_frac_naive" in answers:
    _promoted = answers["noise_frac_naive"] > 0.6 and answers["noise_frac_raw"] < 1e-6
    _reg_ok = answers["noise_frac_reg"] < 1e-3
    _white = answers["cov_err_trunc"] < 1e-6
    print(f"  {'noise promoted by naive':<24} {'ok' if _promoted else 'wrong'}  "
          f"(raw {answers['noise_frac_raw']:.2e} -> naive {answers['noise_frac_naive']:.3f})")
    print(f"  {'regularized suppresses it':<24} {'ok' if _reg_ok else 'wrong'}  "
          f"({answers['noise_frac_reg']:.2e})")
    print(f"  {'truncated is white':<24} {'ok' if _white else 'wrong'}")
    _ok = _ok and _promoted and _reg_ok and _white

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The tail eigenvalues here are around `1e-12`, so dividing by `sqrt(lambda)`
multiplies those directions by about `1e6`, and since the noise is genuinely
that small, the result is not an overflow but something worse to debug: 32
perfectly well-scaled unit-variance features made entirely of rounding error.
Adding `eps` inside the square root caps the gain at `1/sqrt(eps)`, leaving the
noise directions near zero, and truncating removes them outright. Truncation is
the honest choice when the data really is low rank. Regularization is the one
to reach for when you are not sure.

```python
import numpy as np

Xc = X - X.mean(axis=0)
C = (Xc.T @ Xc) / (len(Xc) - 1)
w, V = np.linalg.eigh(C)
order = np.argsort(w)[::-1]
w, V = w[order], V[:, order]

P = Xc @ V

naive_white = P / np.sqrt(w)
reg_white = P / np.sqrt(w + 1e-5)

keep = w > 1e-8
trunc_white = P[:, keep] / np.sqrt(w[keep])

Ct = np.cov(trunc_white, rowvar=False)

def noise_frac(Z):
    v = Z.var(axis=0)
    return float(v[8:].sum() / v.sum())

answers = {
    "eigvals": w,
    "n_kept": int(keep.sum()),
    "noise_frac_raw": noise_frac(P),
    "noise_frac_naive": noise_frac(naive_white),
    "noise_frac_reg": noise_frac(reg_white),
    "cov_err_trunc": float(np.linalg.norm(Ct - np.eye(int(keep.sum())))),
}
```